# Preliminary Analysis

Derives the per-sensor settings used by `01_regression`:

- Calculates noise level from zero-concentration baselines.
- Establishes the best data representation using feature ablation.
- Determines PCA components needed using 95% cumulative EVR.

Every visualisation is fitted on the full
non-zero dataset and is exploratory only.

Exported to `preliminary_results.pkl`.

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# paths
FILE_PATH       = 'processed_data.pkl'
PRELIMINARY_OUT = 'preliminary_results.pkl'

# constants
RANDOM_STATE    = 42
MAX_COMPONENTS  = 50    # hard cap on PCA components explored
PROBE_COMPONENTS= 20    # components used in representation ablation probe

# optional overrides (None -> use per-sensor winner from ablation)
COMMON_REPR = None   # e.g. 'full'
COMMON_COMP = None   # e.g. 20

# frequency axis
N_FREQ_PTS = 6401
FREQ_GHZ   = np.linspace(2, 8, N_FREQ_PTS)

plt.rcParams.update({
    'font.size'         : 10,
    'figure.figsize'    : (10, 6),
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

SENSOR_CONFIG = {
    'Sensor A — GO/Nafion (S11)'      : 'features_a',
    'Sensor B — G/GO/PEDOT:PSS (S22)' : 'features_b',
}

# Journal style (IEEE Transactions two-column)
# Single column: 3.5 in (88.9 mm) | Double column: 7.16 in (181.8 mm)
# For Elsevier single/double: ~90 mm / 190 mm. Wiley: similar.
SINGLE_COL = 3.5    # inches
DOUBLE_COL = 7.16   # inches

plt.rcParams.update({
    'font.family':         'sans-serif',
    # 'font.family':         'monospace',
    # 'font.monospace':      ['Monaspace Neon', 'DejaVu Sans Mono', 'Courier New', 'Consolas'],
    'font.sans-serif':     ['DejaVu Sans', 'Lucida Grande', 'Verdana'],
    'font.serif':          ['Times New Roman', 'DejaVu Serif'],
    'font.size':           8,       # IEEE minimum is 8 pt
    'axes.labelsize':      8,
    'axes.titlesize':      8,
    'xtick.labelsize':     7,
    'ytick.labelsize':     7,
    'legend.fontsize':     7,
    'legend.framealpha':   0.9,
    'legend.edgecolor':    '0.8',
    'legend.handlelength': 1.5,
    'axes.linewidth':      0.5,
    'xtick.major.width':   0.5,
    'ytick.major.width':   0.5,
    'lines.linewidth':     1.0,
    'patch.linewidth':     0.5,
    'axes.grid':           True,
    'grid.linewidth':      0.3,
    'grid.alpha':          0.4,
    'grid.linestyle':      '--',
    'axes.spines.top':     False,
    'axes.spines.right':   False,
    'figure.dpi':          150,    # screen preview
    'savefig.dpi':         600,    # final submission (line art)
    # 'savefig.bbox':        'tight',
    'savefig.format':      'pdf',  # vector default
})

In [ ]:
with open(FILE_PATH, 'rb') as f:
    df_all = pickle.load(f)
    

from numpy.random import default_rng  

# arr_indices_top_drop = default_rng().choice(df_all.index, size=int(len(df_all)*0.95), replace=False)
arr_indices_top_drop = default_rng().choice(df_all.index, size=int(len(df_all)*0.5), replace=False)
df_all.drop(index=arr_indices_top_drop,inplace=True)


df_base = df_all[df_all['concentration_ul'] == 0].copy().reset_index(drop=True)
df      = df_all[df_all['concentration_ul'] >  0].copy().reset_index(drop=True)

GAS_TYPES  = sorted(df['gas_type'].unique())
N_GASES    = len(GAS_TYPES)
GAS_COLORS = dict(zip(GAS_TYPES, sns.color_palette('tab10', N_GASES)))

print(f'Baseline sweeps (conc=0) : {len(df_base):,}')
print(f'Analysis sweeps (conc>0) : {len(df):,}')
print(f'Gas types ({N_GASES})        : {GAS_TYPES}')

## Noise Floor Estimation

In [ ]:
def estimate_noise_from_baseline(df_base, feature_col, sensor_name):
    """Compute per-frequency std dev from zero-concentration sweeps."""
    X = np.stack(df_base[feature_col].values)
    std_per_freq = X.std(axis=0)
    noise_level  = round(float(np.median(std_per_freq)), 6)
    print(f'[{sensor_name}]  baseline sweeps: {len(X):,}  |  noise level: {noise_level}')
    return noise_level, std_per_freq


noise_a, std_a = estimate_noise_from_baseline(df_base, 'features_a', 'Sensor A')
noise_b, std_b = estimate_noise_from_baseline(df_base, 'features_b', 'Sensor B')
NOISE_LEVEL = round(float(np.mean([noise_a, noise_b])), 6)
print(f'\nAdopted NOISE_LEVEL = {NOISE_LEVEL}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, SINGLE_COL*0.8), sharey=True)

for ax, feat_col, std_vec, noise, name in zip(
    axes,
    ['features_a', 'features_b'],
    [std_a, std_b],
    [0.02, 0.02],
    ['Sensor A', 'Sensor B'],
):
    X_base  = np.stack(df_base[feat_col].values)
    std_noisy = (X_base + np.random.normal(0, noise, X_base.shape)).std(axis=0)

    ax.fill_between(FREQ_GHZ, -std_vec,   std_vec,   color='steelblue', alpha=0.25,
                    label='noise baseline')
    ax.fill_between(FREQ_GHZ, -std_noisy, std_noisy, color='tomato',    alpha=0.20,
                    label=f'noise injection (σ={noise:.3f})')
    ax.set_ylim(-0.15,0.15)
    ax.set_xlabel('Frequency (GHz)')
    ax.set_title(f'{name}')
    ax.legend(fontsize=7, loc='upper left', ncol=1)

axes[0].set_ylabel('Noise Level')

plt.tight_layout()
plt.savefig('01_preliminary/fig_noise_floor.pdf')
plt.show()

## Feature Representation/Ablation Study

In [ ]:
_i1 = int(N_FREQ_PTS * (6000 - 2000) / (8000 - 2000))   # 2-6 GHz upper index


def build_feature_representations(X_raw: np.ndarray) -> dict:
    return {
        'full'        : X_raw,
        'band_limited': X_raw[:, :_i1],
        'derivative'  : np.gradient(X_raw, axis=1),
    }


def build_representation(X_raw: np.ndarray, repr_name: str) -> np.ndarray:
    reps = build_feature_representations(X_raw)
    if repr_name not in reps:
        raise ValueError(f"Unknown representation '{repr_name}'. "
                         f"Choose from {list(reps.keys())}")
    return reps[repr_name]


cv_probe = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def ablate_representations(df, feature_col, sensor_name):
    """Rank representations by 5-fold CV accuracy on gas_type (PCA + SVC probe)."""
    X_raw = np.stack(df[feature_col].values)
    y_gas = df['gas_type'].values
    reprs = build_feature_representations(X_raw)

    results = {}
    for repr_name, X in reprs.items():
        nc = min(PROBE_COMPONENTS, X.shape[1], X.shape[0] - 1)
        X_noisy = X + np.random.normal(0, NOISE_LEVEL, X.shape)
        probe = Pipeline([
            ('scaler', StandardScaler()),
            ('pca',    PCA(n_components=nc, random_state=RANDOM_STATE)),
            ('svc',    SVC(kernel='rbf', random_state=RANDOM_STATE)),
        ])
        acc = cross_val_score(probe, X_noisy, y_gas, cv=cv_probe,
                              scoring='accuracy').mean()
        results[repr_name] = round(float(acc), 4)

    best_repr = max(results, key=results.__getitem__)
    print(f'[{sensor_name}]  5-fold CV accuracy: {results}  →  best: {best_repr}')
    return results, best_repr


ablation_a, best_repr_a = ablate_representations(df, 'features_a', 'Sensor A')
ablation_b, best_repr_b = ablate_representations(df, 'features_b', 'Sensor B')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, SINGLE_COL*0.8), sharey=True)
for ax, ablation, best, name in zip(
    axes,
    [ablation_a, ablation_b],
    [best_repr_a, best_repr_b],
    ['Sensor A', 'Sensor B'],
):
    colors = ['#0072B2' if k == best else '#BBBBBB' for k in ablation]
    ax.bar(list(ablation.keys()), list(ablation.values()),
           color=colors, edgecolor='black', linewidth=0.5)
    ax.set_ylabel('CV accuracy (PCA+SVC probe)')
    ax.set_ylim(0, 1.05)
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('Feature representation ablation (5-fold CV, gas_type)', y=1.02)
plt.tight_layout()
plt.savefig('01_preliminary/fig_repr_ablation.pdf')
plt.show()

## PCA dimensionality Quantification

In [ ]:
def select_n_components(df, feature_col, best_repr, sensor_name):
    X_raw   = np.stack(df[feature_col].values)
    X       = build_representation(X_raw, best_repr)
    X_noisy = X #+ np.random.normal(0, NOISE_LEVEL, X.shape)

    nc  = min(MAX_COMPONENTS, X.shape[1], len(X) - 1)
    pca = PCA(n_components=nc, random_state=RANDOM_STATE)
    pca.fit(StandardScaler().fit_transform(X_noisy))

    cumvar = np.cumsum(pca.explained_variance_ratio_)
    idx    = np.searchsorted(cumvar, 0.95)
    n_95   = int(min(idx + 1, nc))   # clamp to fitted components
    print(f'[{sensor_name}]  components for 95% EVR: {n_95}  (fitted up to {nc})')
    return n_95, cumvar, pca.explained_variance_ratio_


n_comp_a, cumvar_a, evr_a = select_n_components(df, 'features_a', best_repr_a, 'Sensor A')
n_comp_b, cumvar_b, evr_b = select_n_components(df, 'features_b', best_repr_b, 'Sensor B')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, SINGLE_COL*0.8))

all_axes = []

for ax, cumvar, evr, n_comp, name in zip(
    axes,
    [cumvar_a, cumvar_b],
    [evr_a, evr_b],
    [n_comp_a, n_comp_b],
    ['Sensor A', 'Sensor B'],
):
    xs = np.arange(1, len(evr) + 1)
    ax.bar(xs, evr, alpha=0.65, color='#0072B2', label='Individual EVR')
    ax2 = ax.twinx()
    all_axes.append(ax)
    all_axes.append(ax2)
    ax2.plot(xs, cumvar, color='#E69F00', lw=1.5, label='Cumulative EVR')
    ax2.axhline(0.95, color='red', ls='--', lw=0.9, label='95% threshold')
    ax2.axvline(n_comp, color='grey', ls=':', lw=0.8)
    ax2.annotate(f'n={n_comp}', xy=(n_comp + 0.4, 0.88),
                 fontsize=7, color='grey', va='bottom')
    ax2.set_ylim(0, 1.05)
    ax.set_xlabel('Principal component')
    ax.set_title(name)
    lines  = ax.get_legend_handles_labels()
    lines2 = ax2.get_legend_handles_labels()
    
all_axes[0].sharey(all_axes[2])
all_axes[1].sharey(all_axes[3])
all_axes[2].legend(lines[0] + lines2[0], lines[1] + lines2[1],loc='lower right', fontsize=7)
all_axes[0].set_ylabel('Explained Variance Ratio')
all_axes[3].set_ylabel('Cumulative EVR')
    
plt.tight_layout()
plt.savefig('01_preliminary/fig_evr.pdf')
plt.show()

## Exploratory Visualisation

Projections coloured by `gas_type`, featuring: PCA
2D/3D with scree, smoothed PCA loadings against a reference S11 signal, LDA 2D/3D,
and t-SNE 2D.

In [ ]:
repr_a = COMMON_REPR or best_repr_a
repr_b = COMMON_REPR or best_repr_b
comp_a = COMMON_COMP or n_comp_a
comp_b = COMMON_COMP or n_comp_b

explore_configs = {
    'Sensor A — GO/Nafion (S11)'      : ('features_a', repr_a, comp_a),
    'Sensor B — G/GO/PEDOT:PSS (S22)' : ('features_b', repr_b, comp_b),
}

y_gas   = df['gas_type'].values
gas_c   = [GAS_COLORS[g] for g in y_gas]    # per-sample colour list

# KNN preservation for t-SNE perplexity selection
def knn_preservation_score(X_hi, X_lo, k=10):
    k = min(k, len(X_hi) - 1)
    nh = NearestNeighbors(n_neighbors=k).fit(X_hi).kneighbors(return_distance=False)
    nl = NearestNeighbors(n_neighbors=k).fit(X_lo).kneighbors(return_distance=False)
    return float(np.mean([len(set(nh[i]) & set(nl[i])) / k for i in range(len(X_hi))]))


def select_tsne_perplexity(X_sc, candidates=None, k=10):
    if candidates is None:
        candidates = [p for p in [5, 10, 20, 30, 50] if p < len(X_sc)]
    scores, embs = {}, {}
    for perp in candidates:
        emb = TSNE(n_components=2, perplexity=perp,
                   random_state=RANDOM_STATE, max_iter=1000).fit_transform(X_sc)
        scores[perp] = knn_preservation_score(X_sc, emb, k=k)
        embs[perp]   = emb
    best = max(scores, key=scores.__getitem__)
    print(f'  t-SNE scores: {scores}  →  perplexity={best}')
    return best, embs[best]


tsne_perplexities = {}
print('Explore configs resolved:')
for sn, (fc, rn, nc) in explore_configs.items():
    print(f'  {sn}: repr={rn}, n_comp={nc}')

In [ ]:
from pathlib import Path
Path('01_preliminary').mkdir(exist_ok=True)

from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 (registers 3D projection)

WINDOW = 301
_smooth = np.ones(WINDOW) / WINDOW

for sensor_name, (feat_col, repr_name, n_comp) in explore_configs.items():
    sensor_name = sensor_name[:8]
    sshort = sensor_name.strip().lower().replace(' ', '_')

    print(f'\n=== {sensor_name} ===')

    X_raw   = np.stack(df[feat_col].values)
    X       = build_representation(X_raw, repr_name)
    X_noisy = X + np.random.normal(0, NOISE_LEVEL, X.shape)
    scaler  = StandardScaler()
    X_sc    = scaler.fit_transform(X_noisy)
    freq_x  = FREQ_GHZ[:X.shape[1]]    # may be shorter if band_limited

    # PCA
    nc  = min(n_comp, X_sc.shape[1], len(X_sc) - 1)
    pca = PCA(n_components=nc, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_sc)

    fig = plt.figure(figsize=(DOUBLE_COL, SINGLE_COL * 0.9))

    ax1 = fig.add_subplot(1, 2, 1)

    for gas in GAS_TYPES:
        mask = y_gas == gas
        ax1.scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[GAS_COLORS[gas]], label=gas, s=10, alpha=0.4,
                    edgecolors='none', rasterized=True)

    ax1.set_xlabel('PC 1')
    ax1.set_ylabel('PC 2')
    ax1.set_title(f'PCA 2D — {sensor_name}')
    ax1.legend(title='Gas Type', fontsize=7, loc='center right', bbox_to_anchor=(1.25, 0.5))


    ax2 = fig.add_subplot(1, 2, 2, projection='3d')

    for gas in GAS_TYPES:
        mask = y_gas == gas
        ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
                    c=[GAS_COLORS[gas]], s=8, alpha=0.35) # Legend label removed since 2D handles it

    ax2.set_xlabel('PC 1')
    ax2.set_ylabel('PC 2')
    ax2.set_zlabel('PC 3')
    ax2.set_title(f'PCA 3D — {sensor_name}')

    # Optional: You may still want to slightly zoom the 3D plot so the 
    ax2.set_box_aspect(None, zoom=1.15) 


    plt.tight_layout()

    # no bbox_inches='tight' so the saved size stays exact
    plt.savefig(f'01_preliminary/fig_pca_combined_{sshort}.pdf')
    plt.show()

    # PCA loadings + reference S11
    fig, axes2 = plt.subplots(2, 1, figsize=(DOUBLE_COL, SINGLE_COL * 0.9), sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
    # fig.suptitle(f'Smoothed PCA loadings vs reference S11 - {sensor_name}', fontsize=12)
    for i in range(min(5, nc)):
        smoothed = np.convolve(pca.components_[i], _smooth, mode='same')
        axes2[0].plot(freq_x, smoothed, label=f'PC {i+1}')
    axes2[0].set_ylabel('Loading Weight')
    axes2[0].set_title(f'Smoothed PC Loadings (PC1-5) — {sensor_name}')
    axes2[0].legend(fontsize=7,loc='lower center',ncols=5); axes2[0].grid(True, ls='--', alpha=0.5)
    axes2[1].plot(freq_x, X_raw[0, :X.shape[1]],
                  color='black', label='Reference S11 (Sample 0)')
    axes2[1].set_xlabel('Frequency (GHz)')
    axes2[1].set_ylabel('S11 Magnitude (dB)')
    # axes2[1].set_title(f'Reference S11 Signal - {sensor_name}')
    axes2[1].legend(fontsize=7); axes2[1].grid(True, ls='--', alpha=0.5)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f'01_preliminary/fig_pca_loadings_{sshort}.pdf')
    plt.show()

    # LDA
    print(f'  Fitting LDA...')
    lda    = LinearDiscriminantAnalysis(n_components=min(3, N_GASES - 1))
    X_lda  = lda.fit_transform(X_sc, y_gas)

    fig = plt.figure(figsize=(DOUBLE_COL, SINGLE_COL * 0.9))

    ax1 = fig.add_subplot(1, 2, 1)

    for gas in GAS_TYPES:
        mask = y_gas == gas
        ax1.scatter(X_lda[mask, 0], X_lda[mask, 1],
                    c=[GAS_COLORS[gas]], label=gas, s=10, alpha=0.4,
                    edgecolors='none', rasterized=True)

    ax1.set_xlabel('LD 1')
    ax1.set_ylabel('LD 2')
    ax1.set_title(f'LDA 2D — {sensor_name}')
    ax1.legend(title='Gas Type', fontsize=7, loc='center right', bbox_to_anchor=(1.25, 0.5))


    ax2 = fig.add_subplot(1, 2, 2, projection='3d')

    for gas in GAS_TYPES:
        mask = y_gas == gas
        # 3D LDA needs >= 4 classes for 3 components
        ax2.scatter(X_lda[mask, 0], X_lda[mask, 1], X_lda[mask, 2],
                    c=[GAS_COLORS[gas]], s=8, alpha=0.35) 

    ax2.set_xlabel('LD 1')
    ax2.set_ylabel('LD 2')
    ax2.set_zlabel('LD 3')
    ax2.set_title(f'LDA 3D — {sensor_name}')

    ax2.set_box_aspect(None, zoom=1.15) 


    plt.tight_layout()

    plt.savefig(f'01_preliminary/fig_lda_combined_{sshort}.pdf')
    plt.show()

    # t-SNE
    print(f'  Selecting t-SNE perplexity...')
    X_for_tsne   = X_pca[:, :min(50, nc)]
    best_perp, X_tsne = select_tsne_perplexity(X_for_tsne)
    tsne_perplexities[sensor_name] = best_perp

    fig, ax = plt.subplots(figsize=(SINGLE_COL, SINGLE_COL * 0.9))
    for gas in GAS_TYPES:
        mask = y_gas == gas
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                   c=[GAS_COLORS[gas]], label=gas, s=10, alpha=0.4,
                   edgecolors='none', rasterized=True)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.set_title(f't-SNE (perplexity={best_perp}) — {sensor_name}')
    ax.legend(title='Gas type', fontsize=7, markerscale=2,
              loc='center left', bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.savefig(f'01_preliminary/fig_tsne_{sshort}.pdf')
    plt.show()

## Export Results

In [ ]:
skey_a, skey_b = list(SENSOR_CONFIG.keys())

prelim = {
    'noise_level' : NOISE_LEVEL,
    skey_a : {
        'best_repr'            : best_repr_a,
        'n_components'         : n_comp_a,
        'best_tsne_perplexity' : tsne_perplexities.get(skey_a),
        'cumulative_evr'       : cumvar_a,
        'evr_per_component'    : evr_a,
        'ablation_accuracy'    : ablation_a,
    },
    skey_b : {
        'best_repr'            : best_repr_b,
        'n_components'         : n_comp_b,
        'best_tsne_perplexity' : tsne_perplexities.get(skey_b),
        'cumulative_evr'       : cumvar_b,
        'evr_per_component'    : evr_b,
        'ablation_accuracy'    : ablation_b,
    },
    'explore' : {
        'repr_a'         : repr_a,
        'repr_b'         : repr_b,
        'n_components_a' : comp_a,
        'n_components_b' : comp_b,
    },
}

with open(PRELIMINARY_OUT, 'wb') as f:
    pickle.dump(prelim, f)

print(f'Saved {PRELIMINARY_OUT}')
print(f'  Sensor A : repr={best_repr_a}, n_comp={n_comp_a},'
      f' ablation={ablation_a}')
print(f'  Sensor B : repr={best_repr_b}, n_comp={n_comp_b},'
      f' ablation={ablation_b}')
print(f'  Noise level : {NOISE_LEVEL}')
print('Ready for 01_regression.ipynb')